In [4]:
import gc

import pandas as pd
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from sklearn.model_selection import RepeatedKFold
import default_risk.config as cfg
import os
import xgboost as xgb
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from default_risk.scripts.auxiliars_for_modeling import apply_cyclical_encoding
import joblib
import lightgbm as lgb
from typing import Optional, List
import optuna
from optuna_integration.mlflow import MLflowCallback
import mlflow
import numpy as np
import yaml
from sklearn.model_selection import cross_val_score
import xgboost as xgb
import re


import dtale
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import get_pipeline

from default_risk.scripts.auxiliars_for_modeling import prepare_columns
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi
from default_risk.scripts.feature_cleaner import clean_noise_from_feature_importance
from default_risk.scripts.feature_cleaner import creating_criteria
from optuna_integration.mlflow import MLflowCallback


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)


load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)



In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_test-processed.parquet")
prev_app_df = pd.read_parquet(cfg.PROCESSED_DIR / "previous_application.train-processed.parquet")


merged_df = application_train_df.merge(
    prev_app_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_df
gc.collect()

X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)
model= xgb.XGBClassifier(**hiperparams)
run_cv_tracked_mlflow(model,hiperparams,cv,X,Y,experiment_name,"application_train + prev_application + instalament + credit card + cash balance")

#freeing memory
del merged_df
gc.collect()



In [ ]:
#for the second one  we gonna analize the gains from the aggregation of bureau
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_feature_engineering.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau.train-processed.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

model= xgb.XGBClassifier(**hiperparams)

run_cv_tracked_mlflow(model,hiperparams,cv,X,Y,experiment_name,"application_train + bureau + bureau_balance")



#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding")
prev_app_installment_agg_df = pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")


merged_df = application_train_df.merge(
    prev_app_installment_agg_df, 
    on="id_curr", 
    how="left"
)


del application_train_df, prev_app_installment_agg_df


gc.collect()


X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)




#run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"prev_app+installment fpi",enable_feature_permutation=True)

#importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_max_rows_internal_parent.csv")
#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "max_rows.csv")
#X = clean_noise_from_feature_importance(importance_df,X,0.0025)
#X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.0003)


run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"raw_internal_parent_target_encoding",persist_feature_importance=True)

#cleaned.to_parquet(cfg.PROCESSED_DIR / "pruned_prev_app_with_installments")

#auc_score_OOF= 0.763

#freeing memory
del merged_df
gc.collect()

In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "pipeline_baseline.parquet")
prev_app_installment_agg_df = pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")


merged_df = application_train_df.merge(
    prev_app_installment_agg_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_installment_agg_df


gc.collect()


X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "max_cols_internal.csv")
importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_max_cols_internal.csv")

#X = clean_noise_from_feature_importance(importance_df,X,0.0025)
X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.00010)



run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"max_cols_internal")

#cleaned.to_parquet(cfg.PROCESSED_DIR / "pruned_prev_app_with_installments")

#auc_score_OOF= 0.763

#freeing memory
del merged_df
gc.collect()

In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
prev_app_installment_agg_df = pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")


merged_df = application_train_df.merge(
    prev_app_installment_agg_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_installment_agg_df


gc.collect()


X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

X= apply_cyclical_encoding(X,"hour_appr_process_start_prev_1",24)




X.drop(columns=["hour_appr_process_start_prev_1"],inplace=True)

model= xgb.XGBClassifier(**hiperparams)
categorical_features= ["organization_type","occupation_type","code_reject_reason_prev_1","name_income_type","name_goods_category_prev_1","name_cash_loan_purpose_prev_1"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)#,,"product_combination_prev_1"

#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "internal_parent_target_enconding_max_cols_feature_importance.csv")
importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_internal_parent_target_enconding_max_cols.csv")

#X = clean_noise_from_feature_importance(importance_df,X,0.0024739875)
#X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.0003)

#pd.get_dummies(X,columns= ["name_contract_type"])



run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"internal_parent_target_enconding_max_cols")


#cleaned.to_parquet(cfg.PROCESSED_DIR / "pruned_prev_app_with_installments")

#auc_score_OOF= 0.763

#freeing memory
del merged_df
gc.collect()  

In [ ]:
#now bureau parent (main with feature engineering + Bureau with feature engineering + Bureau_balance)
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "toxic_baseline.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)



#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_bureau_parent.csv")

#X= clean_importance_zero_and_negative_pfi(importance_df,X)
#importance_permutation_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_bureau_parent.csv")
importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "external_importance.csv")
#criteria = creating_criteria(importance_df,importance_permutation_df)

#importance2 = pd.read_csv(cfg.ARTIFACTS_DIR / "second filter.csv") #
#X= X.drop(columns=["bureau_balance_is_delincuency_sum_loan_1","bureau_has_bureau_balance_data_loan_1","ext_source_1_is_missing"]) 

X= clean_noise_from_feature_importance(importance_df,X,0.004)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"external_parent_co_sample_clean")



#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

In [ ]:
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

model= xgb.XGBClassifier(**hiperparams)
categorical_features=  ["organization_type","occupation_type","name_income_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)


#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_external_parent_target_encoding.csv")

#X= clean_importance_zero_and_negative_pfi(importance_df,X,0.0003)



run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"external_parent_target_encoding")



#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments_time_window.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()



X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

feature_raper= pd.read_csv(cfg.ARTIFACTS_DIR / "final_importance.csv")

X= clean_noise_from_feature_importance(feature_raper,X)

merged_df= merged_df.drop(columns=["flag_email"])

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"2.1 (app_train_with_features+bureau+prev_app+installments)")

#auc_score_OOF=  is the result of all the agregation at 2.0

#freeing memory
del merged_df
gc.collect()

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "pipeline_baseline.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)


#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")



merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()





X,Y = prepare_columns(merged_df)

X = cast_object_into_categoricals(X)

#importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_max_cols_internal.csv")




#X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.00010)


run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"max_rows_final_model")

#auc_score_OOF=  0.781

#freeing memory
del merged_df
gc.collect()

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

features_from_internal_historial= pd.read_csv(cfg.ARTIFACTS_DIR / "internal_best_result.csv")
features_from_external_historial= pd.read_csv(cfg.ARTIFACTS_DIR / "external_best_result.csv")

internal_list=  features_from_internal_historial["feature_name"].to_list()
external_list=  features_from_external_historial["feature_name"].to_list()
features_names = list(set(internal_list + external_list))

X,Y = prepare_columns(merged_df)

X= X[features_names]

X= X.drop(columns= ["amt_down_payment_sum"])



X = cast_object_into_categoricals(X)



run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"final_model_from_convination_of_best_results")

#auc_score_OOF=  0.781

#freeing memory
del merged_df
gc.collect()

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

par = {
'n_estimators': 4693,
'learning_rate': 0.007553343326775532,
'num_leaves': 37,
'min_child_samples': 220,
'min_child_weight': 0.0028908609938903796,
"max_depth" :-1,           
'subsample': 0.8389118020044092,    
'subsample_freq': 3,   
'colsample_bytree': 0.5616610992553818,
'reg_alpha': 2.24385145804699e-07,
'reg_lambda': 7.747405452353306,
'min_split_gain': 0.786374413940256, 
"random_state" : 42,
'cat_smooth': 7.237675739015409,
'cat_l2': 75.34738775153713,
"n_jobs" : 6,
"objective" : 'binary',
"force_col_wise": True,
"importance_type" : "gain"
}

best_params_optuna = {'target_enc_smooth': 2.5821099004254604, 'n_estimators': 4693, 'learning_rate': 0.007553343326775532, 'num_leaves': 37, 'min_child_samples': 220, 'min_child_weight': 0.0028908609938903796, 'subsample': 0.8389118020044092, 'subsample_freq': 3, 'colsample_bytree': 0.5616610992553818, 'reg_alpha': 2.24385145804699e-07, 'reg_lambda': 7.747405452353306, 'min_split_gain': 0.786374413940256, 'cat_smooth': 7.237675739015409, 'cat_l2': 75.34738775153713}

Mejores_Hiperparámetros= {'target_enc_smooth': 78.0016624983871, 'n_estimators': 4854, 'learning_rate': 0.005876339314273085, 'num_leaves': 40, 'min_child_samples': 77, 'min_child_weight': 0.0010075425502278205, 'subsample': 0.8379685394911657, 'subsample_freq': 5, 'colsample_bytree': 0.5007314038467754, 'reg_alpha': 5.874001031590365, 'reg_lambda': 2.473060711126495, 'min_split_gain': 0.41384599131751537, 'cat_smooth': 91.59649623757437, 'cat_l2': 28.173404262475664}
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

model_lgbm = lgb.LGBMClassifier(**par)


categorical_features= ["organization_type","occupation_type","bureau_credit_type_loan_1","wallsmaterial_mode"] # #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1", "organization_type" "organization_type",


pipeline= get_pipeline(2.5821099004254604,categorical_features,model_lgbm)





#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df,santize_text=True)






#X["external_ratio_debt_income"]  =  X["bureau_amt_credit_sum_loan_1"] / X["amt_income_total"]







#X["ratio_credit_active_external"] = np.where(X["amt_income_total"],X["active_amt_credit_sum_active_sum"] / X["amt_income_total"],np.nan)



X["ratio_debt_total_external"] = np.where(X["ext_source_mean"].min(),X["active_amt_credit_sum_debt_active_sum"] / X["ext_source_mean"],np.nan)



#X["balance_income_ratio"] = np.where(X["amt_income_total"],X["last_6_credit_card_amt_balance_mean"] / X["amt_income_total"],np.nan)




X["antique_annuity_vs_actual_annuity"] = np.where(X["amt_annuity"],X["amt_annuity_median"] / X["amt_annuity"],np.nan)

X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)




X['random_noise'] = np.random.normal(0, 1, len(X))


X = cast_object_into_categoricals(X)


model=xgb.XGBClassifier(**hiperparams)

importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm.csv")

X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.00001)

snd_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_with_first_cut.csv")

X = clean_importance_zero_and_negative_pfi(snd_filter,X,0.00001)


trd_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_with_third_cut.csv")

X = clean_importance_zero_and_negative_pfi(trd_filter,X,0.00007)

last_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_for_pipeline.csv")

X = clean_importance_zero_and_negative_pfi(last_filter,X,0.00005)


slow_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_last_cut.csv")

X= pd.get_dummies(X, columns= ["education_type"])

X.columns =  [re.sub(r'[^A-Za-z0-9_]', '_', c) for c in X.columns]

slow_filter["feature"] = [re.sub(r'[^A-Za-z0-9_]', '_', c) for c in slow_filter["feature"].to_list()]

#X = clean_importance_zero_and_negative_pfi(slow_filter,X)








run_cv_tracked_mlflow(pipeline,par,cv,X,Y,experiment_name,"lightgbm_last_cut")

#auc_score_OOF=  0.781

#freeing memory
del merged_df
gc.collect()

eliminando ['hour_appr_process_start_prev_1', 'closed_log_amt_credit_sum_closed_mean', 'cnt_payment_max', 'amt_req_credit_breau_mon', 'credit_card_amt_credit_limit_actual_std_prev_1', 'amt_credit_max', 'instalments_amount_of_versions_in_sequence_sum', 'active_credit_type_credit_card_active_sum', 'bureau_ratio_credit_annuity_loan_1', 'log_amt_down_payment_mean', 'active_balance_months_balance_min_active_min', 'housing_type', 'log_amt_down_payment_std', 'amt_application_sum', 'active_ratio_credit_annuity_active_mean', 'name_yield_group_prev_1', 'obs_60_cnt_social_circle', 'amt_goods_price_min', 'closed_balance_months_since_delincuency_closed_max', 'log_total_interest_charged_mean', 'credit_card_cnt_drawings_atm_current_mean_prev_1', 'active_ratio_credit_annuity_active_max', 'credit_card_name_contract_status_active_sum_prev_1', 'instalments_amt_instalment_median_prev_1', 'closed_amt_annuity_closed_mean', 'closed_balance_months_balance_min_closed_min', 'log_amt_credit_std', 'implied_intere

69553

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()
list_to_delete= eliminar_colinealidad(merged_df,0.95)




In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)

#X= clean_colineality(X,list_to_delete)


model= xgb.XGBClassifier(**hiperparams)
categorical_features= ["organization_type","occupation_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
 #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)


X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)

X = cast_object_into_categoricals(X)



#X= X.drop(columns=cols_to_drop)

importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_monster_final_model.csv")

X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,-0.00001)

second_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "second.csv")

X = clean_importance_zero_and_negative_pfi(second_filter,X,-0.00001)

third_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "third.csv")

X = clean_importance_zero_and_negative_pfi(third_filter,X,-0.00001)

fourth_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "fourth.csv")

X = clean_importance_zero_and_negative_pfi(fourth_filter,X,0.00001)



run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"monster_without_co_lineality",enable_feature_permutation=False)

#auc_score_OOF=  0.781

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)

#X= clean_colineality(X,list_to_delete)




model= xgb.XGBClassifier(**hiperparams)
categorical_features= ["organization_type","occupation_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
 #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)


X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)

X = cast_object_into_categoricals(X)

cols_to_drop= ["name_income_type"]

importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_monster_final_model.csv")


X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,-0.00001)

second_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "second.csv")

X = clean_importance_zero_and_negative_pfi(second_filter,X,-0.00001)

third_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "third.csv")

X = clean_importance_zero_and_negative_pfi(third_filter,X,-0.00001)

fourth_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "fourth.csv")

X = clean_importance_zero_and_negative_pfi(fourth_filter,X,0.00001)

X= X.drop(columns=cols_to_drop)






#five_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_fine_pruned_final_model.csv")

#X = clean_importance_zero_and_negative_pfi(five_filter,X,0.00009)

run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"fine_pruned_final_model")


In [12]:
# 1. Función Objetivo (Ahora recibe X, Y, y las categóricas explícitamente)
mlflow_callback = MLflowCallback(
    tracking_uri="mlruns",
    metric_name="roc_auc",
    create_experiment=True
)

def objective(trial, X, Y, categorical_features):


    target_enc_smooth = trial.suggest_float("target_enc_smooth", 1.0, 100.0, log=True)

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 5000),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 10.0, log=True),

        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 7), # Activa el uso de subsample
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),

        "cat_smooth": trial.suggest_float("cat_smooth", 1.0, 100.0, log=True),
        "cat_l2": trial.suggest_float("cat_l2", 1e-2, 100.0, log=True),

        "max_depth": -1, 
        "random_state": 42,
        "n_jobs": 14,
        "objective": 'binary',       
        "force_col_wise": True,
        "importance_type": "gain"
    }

    model = lgb.LGBMClassifier(**params)


    pipeline = get_pipeline(target_enc_smooth, categorical_features, model)
    
    auc_scores = cross_val_score(
        pipeline, 
        X, 
        Y, 
        cv=5, 
        scoring="roc_auc", 
        n_jobs=1
    )
    
    return np.mean(auc_scores)


# 2. Función Principal (Recibe los datos y configura el estudio)
def run_optimization(X_train, Y_train, cat_features):
    study = optuna.create_study(
        study_name="lightgbm_tuning",
        direction="maximize" 
    )
    
    # EL TRUCO: Usamos un lambda para inyectar los datos preservando el 'trial'
    study.optimize(
        lambda trial: objective(trial, X_train, Y_train, cat_features), 
        n_trials=300, 
        callbacks=[mlflow_callback]
    )
    
    return study


    

C:\Users\kuroc\AppData\Local\Temp\ipykernel_1560\2158620581.py:2: FutureWarning: MLflowCallback has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0.
  mlflow_callback = MLflowCallback(


In [13]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

model_lgbm = lgb.LGBMClassifier(**par)


categorical_features= ["organization_type","occupation_type","bureau_credit_type_loan_1","wallsmaterial_mode"] # #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1", "organization_type" "organization_type",


pipeline= get_pipeline(2.5821099004254604,categorical_features,model_lgbm)





#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df,santize_text=True)






#X["external_ratio_debt_income"]  =  X["bureau_amt_credit_sum_loan_1"] / X["amt_income_total"]







#X["ratio_credit_active_external"] = np.where(X["amt_income_total"],X["active_amt_credit_sum_active_sum"] / X["amt_income_total"],np.nan)

#X["ratio_debt_total_external"] = np.where(X["amt_income_total"],X["active_amt_credit_sum_debt_active_sum"] / X["amt_income_total"],np.nan)



#X["balance_income_ratio"] = np.where(X["amt_income_total"],X["last_6_credit_card_amt_balance_mean"] / X["amt_income_total"],np.nan)




X["antique_annuity_vs_actual_annuity"] = np.where(X["amt_annuity"],X["amt_annuity_median"] / X["amt_annuity"],np.nan)

X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)




X['random_noise'] = np.random.normal(0, 1, len(X))


X = cast_object_into_categoricals(X)


model=xgb.XGBClassifier(**hiperparams)

importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm.csv")

X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.00001)

snd_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_with_first_cut.csv")

X = clean_importance_zero_and_negative_pfi(snd_filter,X,0.00001)


trd_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_with_third_cut.csv")

X = clean_importance_zero_and_negative_pfi(trd_filter,X,0.00007)

last_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_for_pipeline.csv")

X = clean_importance_zero_and_negative_pfi(last_filter,X,0.00005)


slow_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_last_cut.csv")

X= pd.get_dummies(X, columns= ["education_type"])

X.columns =  [re.sub(r'[^A-Za-z0-9_]', '_', c) for c in X.columns]

slow_filter["feature"] = [re.sub(r'[^A-Za-z0-9_]', '_', c) for c in slow_filter["feature"].to_list()]

#X = clean_importance_zero_and_negative_pfi(slow_filter,X)





study = run_optimization(X,Y,categorical_features)
print(f"Mejor AUC alcanzado: {study.best_value}")
print(f"Mejores Hiperparámetros: {study.best_params}")

eliminando ['hour_appr_process_start_prev_1', 'closed_log_amt_credit_sum_closed_mean', 'cnt_payment_max', 'amt_req_credit_breau_mon', 'credit_card_amt_credit_limit_actual_std_prev_1', 'amt_credit_max', 'instalments_amount_of_versions_in_sequence_sum', 'active_credit_type_credit_card_active_sum', 'bureau_ratio_credit_annuity_loan_1', 'log_amt_down_payment_mean', 'active_balance_months_balance_min_active_min', 'housing_type', 'log_amt_down_payment_std', 'amt_application_sum', 'active_ratio_credit_annuity_active_mean', 'name_yield_group_prev_1', 'obs_60_cnt_social_circle', 'amt_goods_price_min', 'closed_balance_months_since_delincuency_closed_max', 'log_total_interest_charged_mean', 'credit_card_cnt_drawings_atm_current_mean_prev_1', 'active_ratio_credit_annuity_active_max', 'credit_card_name_contract_status_active_sum_prev_1', 'instalments_amt_instalment_median_prev_1', 'closed_amt_annuity_closed_mean', 'closed_balance_months_balance_min_closed_min', 'log_amt_credit_std', 'implied_intere

[I 2026-07-31 20:22:18,554] A new study created in memory with name: lightgbm_tuning


[LightGBM] [Info] Number of positive: 19843, number of negative: 225933
[LightGBM] [Info] Total Bins 25397
[LightGBM] [Info] Number of data points in the train set: 245776, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080736 -> initscore=-2.432387
[LightGBM] [Info] Start training from score -2.432387
[LightGBM] [Info] Number of positive: 19843, number of negative: 225933
[LightGBM] [Info] Total Bins 25381
[LightGBM] [Info] Number of data points in the train set: 245776, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080736 -> initscore=-2.432387
[LightGBM] [Info] Start training from score -2.432387
[LightGBM] [Info] Number of positive: 19843, number of negative: 225933
[LightGBM] [Info] Total Bins 25387
[LightGBM] [Info] Number of data points in the train set: 245776, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080736 -> initscore=-2.432387
[LightGBM] [Info] Start training from score -2

[I 2026-07-31 20:29:46,007] Trial 0 finished with value: 0.7927907784116517 and parameters: {'target_enc_smooth': 52.32180388182352, 'n_estimators': 2705, 'learning_rate': 0.007789254572930205, 'num_leaves': 233, 'min_child_samples': 152, 'min_child_weight': 2.2130098494977575, 'subsample': 0.5330276454510188, 'subsample_freq': 3, 'colsample_bytree': 0.9930146744927142, 'reg_alpha': 6.764447793940678e-05, 'reg_lambda': 4.985101480816006e-08, 'min_split_gain': 0.8838913829743015, 'cat_smooth': 1.280153231307965, 'cat_l2': 0.7310632184695728}. Best is trial 0 with value: 0.7927907784116517.


[LightGBM] [Info] Number of positive: 19843, number of negative: 225933
[LightGBM] [Info] Total Bins 25397
[LightGBM] [Info] Number of data points in the train set: 245776, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080736 -> initscore=-2.432387
[LightGBM] [Info] Start training from score -2.432387
[LightGBM] [Info] Number of positive: 19843, number of negative: 225933
[LightGBM] [Info] Total Bins 25381
[LightGBM] [Info] Number of data points in the train set: 245776, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080736 -> initscore=-2.432387
[LightGBM] [Info] Start training from score -2.432387
[LightGBM] [Info] Number of positive: 19843, number of negative: 225933
[LightGBM] [Info] Total Bins 25387
[LightGBM] [Info] Number of data points in the train set: 245776, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080736 -> initscore=-2.432387
[LightGBM] [Info] Start training from score -2

...

[LightGBM] [Info] Number of positive: 19843, number of negative: 225933
[LightGBM] [Info] Total Bins 25397
[LightGBM] [Info] Number of data points in the train set: 245776, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080736 -> initscore=-2.432387
[LightGBM] [Info] Start training from score -2.432387
[LightGBM] [Info] Number of positive: 19843, number of negative: 225933
[LightGBM] [Info] Total Bins 25381
[LightGBM] [Info] Number of data points in the train set: 245776, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080736 -> initscore=-2.432387
[LightGBM] [Info] Start training from score -2.432387
[LightGBM] [Info] Number of positive: 19843, number of negative: 225933
[LightGBM] [Info] Total Bins 25387
[LightGBM] [Info] Number of data points in the train set: 245776, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080736 -> initscore=-2.432387
[LightGBM] [Info] Start training from score -2

[I 2026-08-01 21:54:58,931] Trial 298 finished with value: 0.799145383885971 and parameters: {'target_enc_smooth': 56.75576318513503, 'n_estimators': 4996, 'learning_rate': 0.005482048655836533, 'num_leaves': 49, 'min_child_samples': 106, 'min_child_weight': 0.002394549882655214, 'subsample': 0.8266296072739087, 'subsample_freq': 5, 'colsample_bytree': 0.5388277148644939, 'reg_alpha': 3.4534417839013294, 'reg_lambda': 4.2408880625820995, 'min_split_gain': 0.3975161028298059, 'cat_smooth': 72.41891964496035, 'cat_l2': 33.36329755404526}. Best is trial 184 with value: 0.7993204620312081.


[LightGBM] [Info] Number of positive: 19843, number of negative: 225933
[LightGBM] [Info] Total Bins 25397
[LightGBM] [Info] Number of data points in the train set: 245776, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080736 -> initscore=-2.432387
[LightGBM] [Info] Start training from score -2.432387
[LightGBM] [Info] Number of positive: 19843, number of negative: 225933
[LightGBM] [Info] Total Bins 25381
[LightGBM] [Info] Number of data points in the train set: 245776, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080736 -> initscore=-2.432387
[LightGBM] [Info] Start training from score -2.432387
[LightGBM] [Info] Number of positive: 19843, number of negative: 225933
[LightGBM] [Info] Total Bins 25387
[LightGBM] [Info] Number of data points in the train set: 245776, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080736 -> initscore=-2.432387
[LightGBM] [Info] Start training from score -2

[I 2026-08-01 22:00:32,438] Trial 299 finished with value: 0.7990377082858304 and parameters: {'target_enc_smooth': 60.58510564867388, 'n_estimators': 4776, 'learning_rate': 0.005236635311434901, 'num_leaves': 50, 'min_child_samples': 108, 'min_child_weight': 0.002425686275660364, 'subsample': 0.8290959250443867, 'subsample_freq': 5, 'colsample_bytree': 0.5394113600391895, 'reg_alpha': 3.466362246646633, 'reg_lambda': 6.766077293687275, 'min_split_gain': 0.39723682361159546, 'cat_smooth': 66.87011949528188, 'cat_l2': 47.35694612649366}. Best is trial 184 with value: 0.7993204620312081.


Mejor AUC alcanzado: 0.7993204620312081
Mejores Hiperparámetros: {'target_enc_smooth': 78.0016624983871, 'n_estimators': 4854, 'learning_rate': 0.005876339314273085, 'num_leaves': 40, 'min_child_samples': 77, 'min_child_weight': 0.0010075425502278205, 'subsample': 0.8379685394911657, 'subsample_freq': 5, 'colsample_bytree': 0.5007314038467754, 'reg_alpha': 5.874001031590365, 'reg_lambda': 2.473060711126495, 'min_split_gain': 0.41384599131751537, 'cat_smooth': 91.59649623757437, 'cat_l2': 28.173404262475664}


In [ ]:
X.head()

In [ ]:
test_application= pd.read_parquet(cfg.MASTER_DATA_DIR / "prepared_dataset_test.parquet")
dtale.show(test_application.head(5))

Mejores Hiperparámetros: {'n_estimators': 2086, 'learning_rate': 0.010061666728286038, 'num_leaves': 68, 'min_child_samples': 151, 'subsample': 0.8664564642957002, 'colsample_bytree': 0.6347840589896324, 'reg_alpha': 4.905930240549411, 'reg_lambda': 0.006830459057401382, 'min_split_gain': 0.25616917560909797, 'min_child_weight': 0.010819305760005674}


In [3]:
lgbm_features= pd.read_csv(cfg.ARTIFACTS_DIR / "features_lightbm_pipeline.csv")
feature_names= lgbm_features["feature_name"].tolist()
yaml_text = yaml.dump(feature_names, default_flow_style=False)
print(yaml_text)


- education_type_Incomplete higher
- education_type_Lower secondary
- name_type_suite_prev_1
- code_reject_reason_prev_1
- ratio_debt_income
- credit_card_cnt_drawings_current_max_prev_1
- active_credit_type_mortgage_active_mean
- ext_source_1
- instalments_amount_of_versions_in_sequence_max
- active_credit_type_microloan_active_mean
- name_contract_status_canceled_mean
- def_30_cnt_social_circle
- last_6_cash_balance_amount_advanced_payment_sum
- documents_count
- instalments_is_delinquency_sum_sum
- closed_id_curr_closed_count
- education_type_Secondary / secondary special
- last_6_credit_card_desesperation_ratio_mean
- active_amt_credit_max_overdue_active_sum
- amt_req_credit_breau_qrt
- amt_goods_price
- days_employed
- active_credit_type_consumer_credit_active_sum
- closed_balance_is_delincuency_mean_closed_mean
- instalments_extra_instalament_mean_mean
- days_last_phone_change
- active_id_curr_active_count
- amt_annuity
- bureau_amt_credit_sum_loan_1
- active_amt_credit_max_overd

In [ ]:
{'target_enc_smooth': 2.5821099004254604, 'n_estimators': 4693, 'learning_rate': 0.007553343326775532, 'num_leaves': 37, 'min_child_samples': 220, 'min_child_weight': 0.0028908609938903796, 'subsample': 0.8389118020044092, 'subsample_freq': 3, 'colsample_bytree': 0.5616610992553818, 'reg_alpha': 2.24385145804699e-07, 'reg_lambda': 7.747405452353306, 'min_split_gain': 0.786374413940256, 'cat_smooth': 7.237675739015409, 'cat_l2': 75.34738775153713}

['bureau_balance_is_delincuency_sum_loan_1', 'closed_days_credit_update_closed_max', 'instalments_completion_ratio_mean', 'last_365_instalments_days_of_delinquency_mean', 'implied_interest_rate_mean', 'last_365_instalments_is_delinquency_mean', 'active_amt_credit_sum_active_max', 'ratio_credit_to_goods_max', 'ext_source_2', 'active_completetitud_ratio_active_min', 'closed_amt_credit_sum_debt_closed_mean', 'payment_trend', 'active_id_curr_active_count', 'wallsmaterial_mode', 'flag_document_3', 'code_reject_reason_prev_1', 'ext_2_x_3', 'bureau_balance_status_score_mean_loan_1', 'active_amt_annuity_active_std', 'ext_source_3', 'region_raiting_client_city', 'amt_goods_price_max', 'amt_down_payment_sum', 'days_and_insurance_information_are_missing_mean', 'education_type_Secondary / secondary special', 'amt_credit', 'closed_days_credit_update_closed_min', 'name_income_type', 'days_termination_prev_1', 'building_score_mean', 'closed_ratio_credit_annuity_closed_max', 'code_gender', 'bureau_day

In [10]:
import yaml

def load_params(param_name)-> dict:
    if not cfg.MODEL_PARAMS.exists():
        raise FileNotFoundError(
            f"There is are no hyperparams defined in {cfg.MODEL_PARAMS}"
        )
        
    with open(cfg.MODEL_PARAMS, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
        return config.get(param_name, {})

load_params("lgbm")

{'hyperparams': {'n_estimators': 2086,
  'learning_rate': 0.010061666728286038,
  'num_leaves': 68,
  'min_child_samples': 151,
  'max_depth': -1,
  'subsample': 0.8664564642957002,
  'colsample_bytree': 0.6347840589896324,
  'reg_alpha': 4.905930240549411,
  'reg_lambda': 0.006830459057401382,
  'min_split_gain': 0.25616917560909797,
  'min_child_weight': 0.010819305760005674,
  'random_state': 42,
  'n_jobs': -1,
  'objective': 'binary',
  'force_col_wise': True,
  'importance_type': 'gain'},
 'features': ['credit_card_cnt_drawings_current_max_prev_1',
  'name_type_suite_prev_1',
  'active_credit_type_microloan_active_mean',
  'amt_req_credit_breau_qrt',
  'instalments_amount_of_versions_in_sequence_max',
  'active_credit_type_mortgage_active_mean',
  'last_6_cash_balance_amount_advanced_payment_sum',
  'code_reject_reason_prev_1',
  'organization_type',
  'bureau_credit_type_loan_1',
  'education_type',
  'name_contract_status_canceled_mean',
  'credit_card_cnt_instalment_mature_cum

In [16]:
dataset = pd.read_parquet(cfg.MASTER_DATA_DIR / 'prepared_dataset_train.parquet')
print(list(dataset.columns))

['id_curr', 'target', 'name_contract_type', 'code_gender', 'flag_own_car', 'flag_own_realty', 'cnt_children', 'amt_income_total', 'amt_credit', 'amt_annuity', 'amt_goods_price', 'amt_goods_price_is_missing', 'name_type_suite', 'name_type_suite_is_missing', 'name_income_type', 'family_status', 'housing_type', 'region_population', 'days_birth', 'have_sentinel_value_days_employed', 'days_employed', 'days_registration', 'days_id_publish', 'own_car_age', 'own_car_age_is_missing', 'flag_emp_phone', 'flag_cont_mobile', 'flag_phone', 'flag_email', 'occupation_type', 'occupation_type_is_missing', 'cnt_family_members', 'region_raiting_client', 'region_raiting_client_city', 'weekday_appr_process_start', 'hour_apply_start', 'flag_region_not_live', 'flag_region_not_work', 'flag_live_region_not_work', 'flag_not_live_city', 'flag_city_not_work', 'flag_live_city_not_work', 'organization_type', 'ext_source_1', 'ext_source_1_is_missing', 'ext_source_2', 'ext_source_2_is_missing', 'ext_source_3', 'ext_so

In [14]:
print(f"Mejor AUC alcanzado: {study.best_value}")
print(f"Mejores Hiperparámetros: {study.best_params}")

Mejor AUC alcanzado: 0.7993204620312081
Mejores Hiperparámetros: {'target_enc_smooth': 78.0016624983871, 'n_estimators': 4854, 'learning_rate': 0.005876339314273085, 'num_leaves': 40, 'min_child_samples': 77, 'min_child_weight': 0.0010075425502278205, 'subsample': 0.8379685394911657, 'subsample_freq': 5, 'colsample_bytree': 0.5007314038467754, 'reg_alpha': 5.874001031590365, 'reg_lambda': 2.473060711126495, 'min_split_gain': 0.41384599131751537, 'cat_smooth': 91.59649623757437, 'cat_l2': 28.173404262475664}
